In [0]:
%python
%pip install geopandas

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import geopandas as gpd
from pyspark.sql.functions import lit

In [0]:
# Hotel Properties 
df_hotel_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load('/Volumes/cre_catalog/landing/raw_files/hotel_properties.csv')


# Fix column names — Delta doesn't allow spaces or special characters
df_hotel_raw = df_hotel_raw \
    .withColumnRenamed("STREET NUMBER",   "STREET_NUMBER") \
    .withColumnRenamed("STREET NAME",     "STREET_NAME") \
    .withColumnRenamed("Community Board", "COMMUNITY_BOARD") \
    .withColumnRenamed("Council District","COUNCIL_DISTRICT") \
    .withColumnRenamed("Census Tract",    "CENSUS_TRACT") \
    .withColumnRenamed("NTA Name",        "NTA_NAME") \
    .withColumnRenamed("NTA Code2",       "NTA_CODE2")

In [0]:
df_hotel_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.bronze.hotel_properties_raw")

print(f"bronze.hotel_properties_raw: {df_hotel_raw.count():,} rows")

In [0]:
# Restaurant Inspections 
df_rest_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load('/Volumes/cre_catalog/landing/raw_files/nyc_restaurant_inspections.csv')


# Fix column names — Delta doesn't allow spaces
df_rest_raw = df_rest_raw \
    .withColumnRenamed("CUISINE DESCRIPTION",  "CUISINE_DESCRIPTION") \
    .withColumnRenamed("INSPECTION DATE",       "INSPECTION_DATE") \
    .withColumnRenamed("VIOLATION CODE",        "VIOLATION_CODE") \
    .withColumnRenamed("VIOLATION DESCRIPTION", "VIOLATION_DESCRIPTION") \
    .withColumnRenamed("CRITICAL FLAG",         "CRITICAL_FLAG") \
    .withColumnRenamed("GRADE DATE",            "GRADE_DATE") \
    .withColumnRenamed("RECORD DATE",           "RECORD_DATE") \
    .withColumnRenamed("INSPECTION TYPE",       "INSPECTION_TYPE") \
    .withColumnRenamed("Community Board",       "COMMUNITY_BOARD") \
    .withColumnRenamed("Council District",      "COUNCIL_DISTRICT") \
    .withColumnRenamed("Census Tract",          "CENSUS_TRACT")

In [0]:
df_rest_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.bronze.restaurant_inspections_raw")

In [0]:
gdf = gpd.read_file(
    '/Volumes/cre_catalog/landing/raw_files/MAPPLUTO.geojson'
)

In [0]:
%python
# Convert geometry to WKT string for Spark compatibility
gdf['geometry_wkt'] = gdf['geometry'].apply(
    lambda x: x.wkt if x is not None else None
)

# Drop original geometry column (Spark can't handle geometry type)
gdf_flat = gdf.drop(columns=['geometry'])

# Convert to Spark DataFrame
df_pluto = spark.createDataFrame(gdf_flat)

# Write to bronze
df_pluto.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cre_catalog.bronze.mappluto_raw")

In [0]:
%sql
SELECT 
    COUNT(*) as total_pluto,
    COUNT(t.PARID) as matched_records,
    COUNT(*) - COUNT(t.PARID) as unmatched
FROM cre_catalog.bronze.mappluto_raw p
LEFT JOIN cre_catalog.bronze.nyc_tax_raw t
    ON CAST(p.BBL AS STRING) = t.PARID;